# 7.14 — The R-CNN Family

The R-CNN family turns image classification into object detection by making detection a two-stage conversation: first ask **where objects might be**, then classify and refine each proposed region. In this notebook we build the core ideas from scratch with NumPy — region proposals, IoU labels, shared feature maps, RoI pooling, box regression, learned anchors, and non-maximum suppression — so every rectangle and score is inspectable.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build the R-CNN family one idea at a time. Run each cell in order and read the printed intermediate values — every piece of geometry is shown so proposals, pooling, regression, and suppression are not black boxes. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, box geometry, feature maps, and small detector math.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random examples.

▶ What you'll see: the only tools used in the lesson are NumPy and Matplotlib.

### 1. Region proposals and IoU labels

Detection begins by replacing “classify the whole image” with “classify a short list of candidate rectangles.” A proposal is not a final answer; it is a hypothesis that might contain an object. To decide whether a proposal is good enough for training, detectors use intersection-over-union:

$$\operatorname{IoU}(A,B)=\frac{\operatorname{area}(A\cap B)}{\operatorname{area}(A\cup B)}.$$

In [ ]:
def area_w(box_w):
    x1, y1, x2, y2 = box_w
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def iou_w(a_w, b_w):
    ix1_w = max(a_w[0], b_w[0]); iy1_w = max(a_w[1], b_w[1])
    ix2_w = min(a_w[2], b_w[2]); iy2_w = min(a_w[3], b_w[3])
    inter_w = area_w([ix1_w, iy1_w, ix2_w, iy2_w])
    union_w = area_w(a_w) + area_w(b_w) - inter_w
    return inter_w / union_w if union_w > 0 else 0.0

A_w = np.array([0., 0., 3., 3.])
B_w = np.array([1., 1., 4., 4.])
print("area(A):", area_w(A_w), "area(B):", area_w(B_w))
print("IoU(A,B):", round(iou_w(A_w, B_w), 3))
assert round(iou_w(A_w, B_w), 3) == 0.286

▶ What you'll see: two 3×3 boxes overlap by a 2×2 square, giving IoU 4/14 = 0.286.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
for box_w, color_w, name_w in [(A_w, "steelblue", "proposal A"), (B_w, "darkorange", "target B")]:
    ax.add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, lw=2, ec=color_w))
    ax.text(box_w[0] + 0.05, box_w[1] + 0.25, name_w, color=color_w)
ax.set_xlim(-0.5, 4.5); ax.set_ylim(4.5, -0.5); ax.set_aspect("equal")
ax.set_title("1: proposal overlap is the training label"); plt.show()

▶ What you'll see: the proposal covers part, but not enough, of the ground-truth rectangle.

In [ ]:
threshold_w = 0.5
label_w = "positive" if iou_w(A_w, B_w) >= threshold_w else "background / negative"
print("IoU threshold:", threshold_w)
print("training label for A:", label_w)

▶ What you'll see: IoU 0.286 is below 0.5, so this proposal is not a positive object crop.

*Why it's done this way:* IoU divides overlap by total covered area, so it penalizes both boxes that miss the object and boxes that are too loose. A fixed threshold converts continuous geometry into supervised labels: high-overlap proposals teach the foreground classifier and regressor; low-overlap proposals teach the background class so the detector does not hallucinate objects everywhere.

### 2. Shared feature maps and RoI pooling

Original R-CNN cropped every proposal and ran a CNN repeatedly. Fast R-CNN changed the cost structure: run the CNN once on the image, then pool a fixed-size feature for each region from the shared feature map. RoI pooling is the bridge between variable-size boxes and a classifier head that expects one fixed vector.

In [ ]:
F_w = np.arange(1, 26, dtype=float).reshape(5, 5)
roi_w = np.array([1., 1., 5., 5.])
print("feature map F:\n", F_w)
print("RoI coordinates:", roi_w)

▶ What you'll see: a 5×5 feature map and a square RoI covering the lower-right 4×4 feature cells.

In [ ]:
def roi_pool_w(F_w, roi_w, out_h_w=2, out_w_w=2):
    x1_w, y1_w, x2_w, y2_w = roi_w.astype(float)
    pooled_w = np.zeros((out_h_w, out_w_w))
    for yy_w in range(out_h_w):
        for xx_w in range(out_w_w):
            xa_w = int(np.floor(x1_w + (x2_w - x1_w) * xx_w / out_w_w))
            xb_w = int(np.ceil(x1_w + (x2_w - x1_w) * (xx_w + 1) / out_w_w))
            ya_w = int(np.floor(y1_w + (y2_w - y1_w) * yy_w / out_h_w))
            yb_w = int(np.ceil(y1_w + (y2_w - y1_w) * (yy_w + 1) / out_h_w))
            patch_w = F_w[ya_w:yb_w, xa_w:xb_w]
            pooled_w[yy_w, xx_w] = np.max(patch_w)
    return pooled_w

pooled_w = roi_pool_w(F_w, roi_w)
print("2×2 RoI pooled feature:\n", pooled_w)
assert np.allclose(pooled_w, [[13, 15], [23, 25]])

▶ What you'll see: each output bin keeps the maximum activation from its slice of the RoI.

In [ ]:
plt.figure(figsize=(4.2, 3.4))
plt.imshow(F_w, cmap="viridis")
plt.colorbar(label="feature value")
plt.gca().add_patch(plt.Rectangle((roi_w[0]-0.5, roi_w[1]-0.5), roi_w[2]-roi_w[0], roi_w[3]-roi_w[1], fill=False, ec="red", lw=2))
plt.title("2: one shared feature map, many RoIs"); plt.show()

▶ What you'll see: the RoI is a rectangle on the feature map, not a separate resized image.

*Why it's done this way:* the convolutional map is expensive, so sharing it avoids recomputing the same visual features for thousands of boxes. Max pooling preserves the strongest local evidence in each bin, and the fixed 2×2 (or 7×7 in real models) output gives the classifier a constant-length input even when proposals have different sizes.

### 3. Region classification, including background

Once a proposal has a fixed feature vector, the second stage scores object classes plus a background class. Background is essential: most candidate boxes are not objects, and the classifier needs an explicit “none of the above” output.

In [ ]:
h_w = pooled_w.ravel()
Wc_w = np.array([[ 0.02,  0.01,  0.01,  0.00],
                 [-0.02, -0.01,  0.03,  0.04],
                 [ 0.01,  0.00, -0.02, -0.01]])
bc_w = np.array([0.1, -0.2, 0.0])
logits_w = Wc_w @ h_w + bc_w
print("pooled vector h:", h_w)
print("class logits [background, cat, dog]:", np.round(logits_w, 3))

▶ What you'll see: the same region feature becomes three raw class scores.

In [ ]:
def softmax_w(z_w):
    z_w = z_w - np.max(z_w)
    exp_w = np.exp(z_w)
    return exp_w / exp_w.sum()

probs_w = softmax_w(logits_w)
classes_w = np.array(["background", "cat", "dog"])
print("probabilities:", dict(zip(classes_w, np.round(probs_w, 3))))
print("predicted class:", classes_w[int(np.argmax(probs_w))])
assert round(float(probs_w.sum()), 6) == 1.0

▶ What you'll see: softmax turns arbitrary logits into probabilities that sum to 1.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(classes_w, probs_w, color=["gray", "teal", "orange"])
plt.ylim(0, 1); plt.ylabel("probability")
plt.title("3: each proposal gets a class distribution"); plt.show()

▶ What you'll see: one bar is highest; if background wins, the proposal is discarded.

*Why it's done this way:* classification is separated from proposal generation because a box can be geometrically plausible yet semantically empty. Softmax normalizes competing class evidence, while the background class absorbs regions that should not be forced into a foreground label.

### 4. Box regression refines rough proposals

The classifier says what the region looks like; the regressor moves the box so it fits the object better. In this lesson we use a simple corner-coordinate correction, $\hat b=r+\Delta b$, so the arithmetic is visible.

In [ ]:
proposal_w = np.array([0., 0., 3., 3.])
target_w = np.array([1., 1., 4., 4.])
delta_w = target_w - proposal_w
refined_w = proposal_w + delta_w
print("proposal:", proposal_w)
print("delta:", delta_w)
print("refined:", refined_w)
assert np.allclose(refined_w, target_w)

▶ What you'll see: adding `[1, 1, 1, 1]` shifts the rough proposal exactly onto the target.

In [ ]:
before_iou_w = iou_w(proposal_w, target_w)
after_iou_w = iou_w(refined_w, target_w)
print("IoU before refinement:", round(before_iou_w, 3))
print("IoU after refinement:", round(after_iou_w, 3))
assert round(after_iou_w, 3) == 1.0

▶ What you'll see: regression improves IoU from 0.286 to 1.000 in this toy corner-coordinate example.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
for box_w, color_w, name_w in [(proposal_w, "steelblue", "before"), (target_w, "black", "target"), (refined_w, "seagreen", "after")]:
    ax.add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, lw=2, ec=color_w))
    ax.text(box_w[0] + 0.05, box_w[1] + 0.25, name_w, color=color_w)
ax.set_xlim(-0.5, 4.5); ax.set_ylim(4.5, -0.5); ax.set_aspect("equal")
ax.set_title("4: regression makes the box worthy of the class"); plt.show()

▶ What you'll see: the refined green box sits on the target instead of the rough blue proposal.

*Why it's done this way:* proposal methods are designed for recall, not perfect localization. Regression lets the detector use visual evidence inside the RoI to predict a small geometric correction, turning “near the object” into “tightly around the object.”

### 5. Faster R-CNN learns proposals with anchors

Faster R-CNN replaces hand-built proposal methods with a region proposal network (RPN). At each feature-map location, it scores several anchor boxes for objectness and regresses anchor corrections. The anchor with the best IoU becomes the clearest positive training example.

In [ ]:
gt_w = np.array([1., 1., 3., 3.])
anchors_w = np.array([[0., 0., 2., 2.],
                      [1., 1., 4., 4.],
                      [0., 0., 3., 3.]])
ious_w = np.array([iou_w(a_w, gt_w) for a_w in anchors_w])
print("anchor IoUs:", np.round(ious_w, 3))
print("first best anchor index:", int(np.argmax(ious_w)))
assert np.allclose(np.round(ious_w, 3), [0.143, 0.444, 0.444])

▶ What you'll see: anchors 1 and 2 tie at IoU 0.444, so NumPy returns the first best index, 1.

In [ ]:
objectness_logits_w = np.array([-1.0, 1.2, 0.7])
objectness_w = 1 / (1 + np.exp(-objectness_logits_w))
print("objectness probabilities:", np.round(objectness_w, 3))
print("top proposal by objectness:", int(np.argmax(objectness_w)))

▶ What you'll see: the RPN can score anchors as likely objects before the second-stage classifier names them.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.add_patch(plt.Rectangle((gt_w[0], gt_w[1]), gt_w[2]-gt_w[0], gt_w[3]-gt_w[1], fill=False, lw=3, ec="black"))
for k_w, box_w in enumerate(anchors_w):
    ax.add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, lw=1.8, ec=plt.cm.tab10(k_w)))
    ax.text(box_w[0] + 0.05, box_w[1] + 0.2, f"a{k_w}: {ious_w[k_w]:.3f}")
ax.set_xlim(-0.5, 4.5); ax.set_ylim(4.5, -0.5); ax.set_aspect("equal")
ax.set_title("5: anchors compete to explain the ground truth"); plt.show()

▶ What you'll see: several predefined rectangles overlap the same target by different amounts.

*Why it's done this way:* anchors discretize the huge space of possible boxes into a small menu of candidate shapes. The RPN learns which anchor locations look object-like and how to adjust them, so Faster R-CNN keeps the two-stage proposal-and-refine logic while making the proposal stage trainable.

### 6. NMS removes duplicate detections

A second-stage classifier often fires on several overlapping proposals around the same object. Non-maximum suppression (NMS) sorts detections by score, keeps the best one, and suppresses lower-scoring boxes whose IoU with a kept box is too high.

In [ ]:
boxes_w = np.array([[0., 0., 3., 3.],
                    [0.5, 0.5, 3.5, 3.5],
                    [5., 5., 7., 7.]])
scores_w = np.array([0.9, 0.8, 0.7])
print("scores:", scores_w)
print("IoU(box0, box1):", round(iou_w(boxes_w[0], boxes_w[1]), 3))
assert round(iou_w(boxes_w[0], boxes_w[1]), 3) == 0.532

▶ What you'll see: boxes 0 and 1 overlap strongly, so they likely describe the same object.

In [ ]:
def nms_w(boxes_w, scores_w, thresh_w=0.3):
    order_w = list(np.argsort(scores_w)[::-1])
    keep_w = []
    while order_w:
        current_w = order_w.pop(0)
        keep_w.append(current_w)
        order_w = [j_w for j_w in order_w if iou_w(boxes_w[current_w], boxes_w[j_w]) <= thresh_w]
    return keep_w

keep_w = nms_w(boxes_w, scores_w, 0.3)
print("kept indices:", keep_w)
assert keep_w == [0, 2]

▶ What you'll see: box 1 is suppressed as a duplicate of box 0, while distant box 2 remains.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
for k_w, box_w in enumerate(boxes_w):
    color_w = "seagreen" if k_w in keep_w else "crimson"
    ax.add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, lw=2, ec=color_w))
    ax.text(box_w[0] + 0.05, box_w[1] + 0.25, f"{k_w}: {scores_w[k_w]}", color=color_w)
ax.set_xlim(-0.5, 7.5); ax.set_ylim(7.5, -0.5); ax.set_aspect("equal")
ax.set_title("6: NMS keeps high-score non-duplicates"); plt.show()

▶ What you'll see: green boxes survive and the red overlapping duplicate is removed.

*Why it's done this way:* detection outputs are not independent; nearby proposals borrow the same visual evidence. NMS imposes a geometric uniqueness rule after scoring, so the final prediction list contains one strong detection per object instead of many near-identical rectangles.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, feature maps, box geometry, and detector arithmetic.
import matplotlib.pyplot as plt  # load Matplotlib for every box, heatmap, and score visualization.
np.random.seed(0)  # make all examples reproducible across notebook runs.

▶ What you'll see: the shared imports for all worked examples below.

## 🟢 Basics (warm-up)

### Basic 1 — Draw one proposal box

**Goal.** Represent a candidate region as `[x1, y1, x2, y2]`, because R-CNN-style detectors reason about rectangles before they reason about classes. We build it in 2 steps.

In [ ]:
box_b1 = np.array([1., 1., 4., 3.])  # Store the proposal as left, top, right, bottom coordinates.
width_b1 = box_b1[2] - box_b1[0]  # Compute width from right minus left.
height_b1 = box_b1[3] - box_b1[1]  # Compute height from bottom minus top.
print("box:", box_b1, "width:", width_b1, "height:", height_b1)
assert width_b1 == 3 and height_b1 == 2

▶ What you'll see: the proposal is three units wide and two units tall.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.add_patch(plt.Rectangle((box_b1[0], box_b1[1]), width_b1, height_b1, fill=False, lw=2, ec="steelblue"))
ax.set_xlim(0, 5); ax.set_ylim(4, 0); ax.set_aspect("equal")
ax.set_title("Basic 1: one candidate region"); plt.show()

▶ What you'll see: a rectangle marking the region that later stages would classify.

👀 Takeaway: a proposal is just geometry until a classifier and regressor act on it.

### Basic 2 — Compute box area

**Goal.** Calculate rectangle area, because IoU, NMS, and training labels all depend on geometric overlap. We build it in 2 steps.

In [ ]:
box_b2 = np.array([0., 0., 3., 3.])
w_b2 = max(0.0, box_b2[2] - box_b2[0])
h_b2 = max(0.0, box_b2[3] - box_b2[1])
area_b2 = w_b2 * h_b2
print("width:", w_b2, "height:", h_b2, "area:", area_b2)
assert area_b2 == 9.0

▶ What you'll see: a 3×3 box has area 9.

In [ ]:
plt.figure(figsize=(3.5, 3))
plt.bar(["width", "height", "area"], [w_b2, h_b2, area_b2], color="teal")
plt.title("Basic 2: area ingredients"); plt.show()

▶ What you'll see: area grows multiplicatively from width and height, not additively.

👀 Takeaway: robust area code clamps negative widths and heights to zero for non-overlapping intersections.

### Basic 3 — Find an intersection rectangle

**Goal.** Find where two boxes overlap, because intersection area is the numerator of IoU. We build it in 2 steps.

In [ ]:
A_b3 = np.array([0., 0., 3., 3.])
B_b3 = np.array([1., 1., 4., 4.])
inter_b3 = np.array([max(A_b3[0], B_b3[0]), max(A_b3[1], B_b3[1]), min(A_b3[2], B_b3[2]), min(A_b3[3], B_b3[3])])
print("intersection box:", inter_b3)
assert np.allclose(inter_b3, [1, 1, 3, 3])

▶ What you'll see: the overlap rectangle runs from `(1,1)` to `(3,3)`.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
for b_b3, c_b3 in [(A_b3, "steelblue"), (B_b3, "darkorange"), (inter_b3, "seagreen")]:
    ax.add_patch(plt.Rectangle((b_b3[0], b_b3[1]), b_b3[2]-b_b3[0], b_b3[3]-b_b3[1], fill=False, lw=2, ec=c_b3))
ax.set_xlim(-0.5, 4.5); ax.set_ylim(4.5, -0.5); ax.set_aspect("equal")
ax.set_title("Basic 3: green is the intersection"); plt.show()

▶ What you'll see: the green rectangle is exactly the shared part of the two boxes.

👀 Takeaway: an intersection box uses maximum left/top and minimum right/bottom coordinates.

### Basic 4 — Compute IoU by hand

**Goal.** Combine intersection and union into IoU, because overlap thresholds decide positives, negatives, and duplicates. We build it in 2 steps.

In [ ]:
A_b4 = np.array([0., 0., 3., 3.])
B_b4 = np.array([1., 1., 4., 4.])
inter_area_b4 = 4.0
union_b4 = 9.0 + 9.0 - inter_area_b4
iou_b4 = inter_area_b4 / union_b4
print("intersection:", inter_area_b4, "union:", union_b4, "IoU:", round(iou_b4, 3))
assert round(iou_b4, 3) == 0.286

▶ What you'll see: union is 14 because the intersection must not be counted twice.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["intersection", "union", "IoU"], [inter_area_b4, union_b4, iou_b4], color=["seagreen", "gray", "purple"])
plt.title("Basic 4: IoU = intersection / union"); plt.show()

▶ What you'll see: IoU is small because the shared area is much smaller than the union.

👀 Takeaway: IoU is a normalized overlap score, so it stays comparable across box sizes.

### Basic 5 — Label a proposal by IoU threshold

**Goal.** Turn IoU into a foreground/background label, because the second-stage classifier needs supervised examples. We build it in 2 steps.

In [ ]:
iou_b5 = 4.0 / 14.0
threshold_b5 = 0.5
is_positive_b5 = iou_b5 >= threshold_b5
print("IoU:", round(iou_b5, 3), "threshold:", threshold_b5)
print("positive proposal?", is_positive_b5)
assert is_positive_b5 == False

▶ What you'll see: the proposal is below a strict 0.5 positive threshold.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["proposal IoU", "positive threshold"], [iou_b5, threshold_b5], color=["orange", "black"])
plt.ylim(0, 1); plt.title("Basic 5: IoU label rule"); plt.show()

▶ What you'll see: the proposal bar sits under the threshold bar.

👀 Takeaway: R-CNN training treats loose boxes as background or ambiguous, not as clean object crops.

### Basic 6 — Pool one RoI to one value

**Goal.** Reduce a variable region to a fixed-size feature, because classifier heads require fixed-length inputs. We build it in 2 steps.

In [ ]:
crop_b6 = np.array([[1., 3.], [2., 4.]])
max_pool_b6 = np.max(crop_b6)
avg_pool_b6 = np.mean(crop_b6)
print("crop:\n", crop_b6)
print("max pool:", max_pool_b6, "average pool:", avg_pool_b6)
assert max_pool_b6 == 4 and avg_pool_b6 == 2.5

▶ What you'll see: max pooling keeps the strongest activation, while average pooling keeps the mean evidence.

In [ ]:
plt.figure(figsize=(3.8, 3))
plt.imshow(crop_b6, cmap="viridis")
plt.colorbar(label="feature")
plt.title("Basic 6: pooled crop values"); plt.show()

▶ What you'll see: the brightest cell is the value max pooling sends forward.

👀 Takeaway: RoI pooling makes differently sized regions compatible with one classifier head.

### Basic 7 — Flatten a pooled feature

**Goal.** Convert a pooled grid into a vector, because the classification and regression heads are linear layers over fixed features. We build it in 2 steps.

In [ ]:
pooled_b7 = np.array([[13., 15.], [23., 25.]])
h_b7 = pooled_b7.ravel()
print("pooled grid:\n", pooled_b7)
print("flattened h:", h_b7)
assert h_b7.shape == (4,)

▶ What you'll see: a 2×2 pooled region becomes a length-4 feature vector.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["h0", "h1", "h2", "h3"], h_b7, color="slateblue")
plt.title("Basic 7: fixed-length region feature"); plt.show()

▶ What you'll see: each pooled bin is now one coordinate in the region descriptor.

👀 Takeaway: fixed vectors let one classifier process every proposal regardless of original box shape.

### Basic 8 — Softmax class scores

**Goal.** Normalize class logits into probabilities, because a detector must choose among background and foreground classes. We build it in 2 steps.

In [ ]:
logits_b8 = np.array([0.1, 1.2, -0.3])
shifted_b8 = logits_b8 - np.max(logits_b8)
probs_b8 = np.exp(shifted_b8) / np.exp(shifted_b8).sum()
print("logits:", logits_b8)
print("probabilities:", np.round(probs_b8, 3))
assert round(float(probs_b8.sum()), 6) == 1.0

▶ What you'll see: probabilities are positive and sum to exactly 1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["bg", "cat", "dog"], probs_b8, color=["gray", "teal", "orange"])
plt.ylim(0, 1); plt.title("Basic 8: softmax over classes"); plt.show()

▶ What you'll see: the largest logit becomes the largest probability.

👀 Takeaway: every proposal is classified against a background alternative.

### Basic 9 — Add a box regression delta

**Goal.** Apply a coordinate correction, because proposals are often close but not tight enough. We build it in 2 steps.

In [ ]:
proposal_b9 = np.array([0., 0., 3., 3.])
delta_b9 = np.array([1., 1., 1., 1.])
refined_b9 = proposal_b9 + delta_b9
print("proposal:", proposal_b9, "delta:", delta_b9, "refined:", refined_b9)
assert np.allclose(refined_b9, [1, 1, 4, 4])

▶ What you'll see: the same offset is added to all four corner coordinates.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
for box_b9, color_b9 in [(proposal_b9, "steelblue"), (refined_b9, "seagreen")]:
    ax.add_patch(plt.Rectangle((box_b9[0], box_b9[1]), box_b9[2]-box_b9[0], box_b9[3]-box_b9[1], fill=False, lw=2, ec=color_b9))
ax.set_xlim(-0.5, 4.5); ax.set_ylim(4.5, -0.5); ax.set_aspect("equal")
ax.set_title("Basic 9: proposal plus regression delta"); plt.show()

▶ What you'll see: the refined box shifts down and right from the original proposal.

👀 Takeaway: box regression is the localization half of the second-stage head.

### Basic 10 — Suppress one duplicate

**Goal.** Compare a high-score detection with a lower-score overlapping detection, because NMS removes duplicates after scoring. We build it in 2 steps.

In [ ]:
box0_b10 = np.array([0., 0., 3., 3.])
box1_b10 = np.array([0.5, 0.5, 3.5, 3.5])
inter_b10 = 2.5 * 2.5
union_b10 = 9.0 + 9.0 - inter_b10
iou_b10 = inter_b10 / union_b10
suppress_b10 = iou_b10 > 0.3
print("IoU:", round(iou_b10, 3), "suppress lower score?", suppress_b10)
assert round(iou_b10, 3) == 0.532

▶ What you'll see: the duplicate box exceeds the 0.3 NMS threshold.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["IoU", "NMS threshold"], [iou_b10, 0.3], color=["crimson", "black"])
plt.ylim(0, 1); plt.title("Basic 10: duplicate suppression rule"); plt.show()

▶ What you'll see: the IoU bar is above the threshold, so the lower-score box is removed.

👀 Takeaway: NMS is part of the detector because region heads naturally produce overlapping duplicates.

## 🟡 Easy

### Easy 1 — Score anchors against one target

**Goal.** Choose the best anchor by IoU, because Faster R-CNN trains its proposal network around anchor/object matches. We build it in 3 steps.

In [ ]:
gt_e1 = np.array([1., 1., 3., 3.])
anchors_e1 = np.array([[0., 0., 2., 2.], [1., 1., 4., 4.], [0., 0., 3., 3.]])
print("ground truth:", gt_e1)
print("anchors:\n", anchors_e1)

▶ What you'll see: three anchor boxes compete to explain the same ground-truth object.

In [ ]:
def area_e1(b_e1):
    return max(0.0, b_e1[2]-b_e1[0]) * max(0.0, b_e1[3]-b_e1[1])
def iou_e1(a_e1, b_e1):
    inter_e1 = [max(a_e1[0], b_e1[0]), max(a_e1[1], b_e1[1]), min(a_e1[2], b_e1[2]), min(a_e1[3], b_e1[3])]
    ia_e1 = area_e1(inter_e1)
    return ia_e1 / (area_e1(a_e1) + area_e1(b_e1) - ia_e1)
ious_e1 = np.array([iou_e1(a_e1, gt_e1) for a_e1 in anchors_e1])
print("IoUs:", np.round(ious_e1, 3))
assert np.allclose(np.round(ious_e1, 3), [0.143, 0.444, 0.444])

▶ What you'll see: the first best anchor is index 1, even though index 2 ties it.

In [ ]:
best_e1 = int(np.argmax(ious_e1))
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1", "a2"], ious_e1, color=["gray", "seagreen", "gray"])
plt.title(f"Easy 1: best anchor = {best_e1}"); plt.ylabel("IoU"); plt.show()

▶ What you'll see: anchor 1 is highlighted as the first maximum-IoU anchor.

👀 Takeaway: RPN training begins by assigning anchors to targets using overlap.

### Easy 2 — Pool several proposals from one feature map

**Goal.** Reuse one feature map for multiple RoIs, because Fast R-CNN is faster than running a CNN per proposal. We build it in 3 steps.

In [ ]:
F_e2 = np.arange(1, 37, dtype=float).reshape(6, 6)
rois_e2 = np.array([[0., 0., 3., 3.], [2., 2., 6., 6.]])
print("feature map shape:", F_e2.shape)
print("RoIs:\n", rois_e2)

▶ What you'll see: two different regions point into the same 6×6 feature map.

In [ ]:
def pool2_e2(F_e2, roi_e2):
    x1_e2, y1_e2, x2_e2, y2_e2 = roi_e2.astype(int)
    crop_e2 = F_e2[y1_e2:y2_e2, x1_e2:x2_e2]
    ys_e2 = np.array_split(np.arange(crop_e2.shape[0]), 2)
    xs_e2 = np.array_split(np.arange(crop_e2.shape[1]), 2)
    out_e2 = np.zeros((2, 2))
    for i_e2, yy_e2 in enumerate(ys_e2):
        for j_e2, xx_e2 in enumerate(xs_e2):
            out_e2[i_e2, j_e2] = np.max(crop_e2[np.ix_(yy_e2, xx_e2)])
    return out_e2
pooled_e2 = np.array([pool2_e2(F_e2, r_e2) for r_e2 in rois_e2])
print("pooled RoIs:\n", pooled_e2)

▶ What you'll see: both uneven source regions become 2×2 arrays.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(F_e2, cmap="viridis")
for r_e2 in rois_e2:
    ax.add_patch(plt.Rectangle((r_e2[0]-0.5, r_e2[1]-0.5), r_e2[2]-r_e2[0], r_e2[3]-r_e2[1], fill=False, lw=2, ec="red"))
ax.set_title("Easy 2: many RoIs share one map"); plt.show()

▶ What you'll see: two red boxes draw from one shared activation grid.

👀 Takeaway: shared convolutional evidence is the efficiency leap from R-CNN to Fast R-CNN.

### Easy 3 — Classify and refine one proposal

**Goal.** Run a tiny second-stage head, because each proposal needs both a class and a coordinate correction. We build it in 3 steps.

In [ ]:
h_e3 = np.array([13., 15., 23., 25.])
W_e3 = np.array([[0.01, 0.00, -0.01, -0.01], [0.00, 0.02, 0.03, 0.01]])
b_e3 = np.array([0.2, -0.1])
logits_e3 = W_e3 @ h_e3 + b_e3
print("logits [background, object]:", np.round(logits_e3, 3))

▶ What you'll see: the object logit is higher than the background logit.

In [ ]:
p_e3 = np.exp(logits_e3 - np.max(logits_e3)); p_e3 = p_e3 / p_e3.sum()
proposal_e3 = np.array([0., 0., 3., 3.])
delta_e3 = np.array([0.5, 0.5, 0.8, 0.8])
refined_e3 = proposal_e3 + delta_e3
print("probabilities:", np.round(p_e3, 3))
print("refined box:", refined_e3)
assert int(np.argmax(p_e3)) == 1

▶ What you'll see: the proposal is classified as object and shifted to a tighter box.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["background", "object"], p_e3, color=["gray", "seagreen"])
plt.ylim(0, 1); plt.title("Easy 3: second-stage class score"); plt.show()

▶ What you'll see: object probability dominates background for this region.

👀 Takeaway: R-CNN heads solve semantic classification and geometric refinement together.

### Easy 4 — Run NMS on three detections

**Goal.** Keep high-score non-overlapping detections, because overlapping proposals often name the same object. We build it in 3 steps.

In [ ]:
boxes_e4 = np.array([[0., 0., 3., 3.], [0.5, 0.5, 3.5, 3.5], [5., 5., 7., 7.]])
scores_e4 = np.array([0.9, 0.8, 0.7])
print("boxes:", boxes_e4.shape, "scores:", scores_e4)

▶ What you'll see: three candidate detections are ready for duplicate removal.

In [ ]:
def area_e4(b_e4):
    return max(0.0, b_e4[2]-b_e4[0]) * max(0.0, b_e4[3]-b_e4[1])
def iou_e4(a_e4, b_e4):
    inter_e4 = [max(a_e4[0], b_e4[0]), max(a_e4[1], b_e4[1]), min(a_e4[2], b_e4[2]), min(a_e4[3], b_e4[3])]
    ia_e4 = area_e4(inter_e4)
    return ia_e4 / (area_e4(a_e4) + area_e4(b_e4) - ia_e4) if area_e4(a_e4) + area_e4(b_e4) - ia_e4 > 0 else 0.0
order_e4 = list(np.argsort(scores_e4)[::-1])
keep_e4 = []
while order_e4:
    cur_e4 = order_e4.pop(0)
    keep_e4.append(cur_e4)
    order_e4 = [j_e4 for j_e4 in order_e4 if iou_e4(boxes_e4[cur_e4], boxes_e4[j_e4]) <= 0.3]
print("kept indices:", keep_e4)
assert keep_e4 == [0, 2]

▶ What you'll see: the overlapping middle box is removed.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["box0", "box1", "box2"], scores_e4, color=["seagreen" if i in keep_e4 else "crimson" for i in range(3)])
plt.title("Easy 4: NMS keeps boxes 0 and 2"); plt.ylabel("score"); plt.show()

▶ What you'll see: kept boxes are green and the suppressed duplicate is red.

👀 Takeaway: NMS turns many region-level predictions into a clean final detection list.

### Easy 5 — Compare R-CNN, Fast R-CNN, and Faster R-CNN cost

**Goal.** Count expensive CNN passes, because the family mainly differs in how it shares features and proposes regions. We build it in 3 steps.

In [ ]:
num_proposals_e5 = 2000
cnn_cost_e5 = 1.0
rcnn_cost_e5 = num_proposals_e5 * cnn_cost_e5
fast_cost_e5 = 1 * cnn_cost_e5
print("R-CNN CNN passes:", rcnn_cost_e5)
print("Fast/Faster R-CNN CNN passes:", fast_cost_e5)
assert rcnn_cost_e5 / fast_cost_e5 == 2000

▶ What you'll see: sharing the feature map reduces the expensive convolutional work by a factor of 2000 in this toy count.

In [ ]:
rpn_extra_e5 = 0.1
costs_e5 = np.array([rcnn_cost_e5, fast_cost_e5, fast_cost_e5 + rpn_extra_e5])
labels_e5 = ["R-CNN", "Fast", "Faster"]
print("relative costs:", dict(zip(labels_e5, costs_e5)))

▶ What you'll see: Faster R-CNN adds a small proposal-network cost but avoids external proposal computation.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(labels_e5, costs_e5, color=["crimson", "teal", "seagreen"])
plt.yscale("log"); plt.ylabel("relative CNN/proposal cost (log)")
plt.title("Easy 5: feature sharing changes runtime"); plt.show()

▶ What you'll see: the R-CNN bar towers over the shared-feature variants on a log scale.

👀 Takeaway: Fast shares features; Faster also learns the proposal stage.

## 🔴 Advanced

### Advanced 1 — Show RoI rounding error

**Goal.** Compare rounded pooling with fractional interpolation, because coarse RoI rounding can shift small objects. We build it in 4 steps.

In [ ]:
F_a1 = np.arange(16, dtype=float).reshape(4, 4)
point_a1 = np.array([1.4, 1.6])
rounded_a1 = F_a1[int(round(point_a1[1])), int(round(point_a1[0]))]
print("feature map:\n", F_a1)
print("fractional point:", point_a1, "rounded sample:", rounded_a1)

▶ What you'll see: rounding jumps the sample to one integer cell.

In [ ]:
x_a1, y_a1 = point_a1
x0_a1, y0_a1 = int(np.floor(x_a1)), int(np.floor(y_a1))
dx_a1, dy_a1 = x_a1 - x0_a1, y_a1 - y0_a1
v00_a1 = F_a1[y0_a1, x0_a1]; v10_a1 = F_a1[y0_a1, x0_a1 + 1]
v01_a1 = F_a1[y0_a1 + 1, x0_a1]; v11_a1 = F_a1[y0_a1 + 1, x0_a1 + 1]
interp_a1 = (1-dx_a1)*(1-dy_a1)*v00_a1 + dx_a1*(1-dy_a1)*v10_a1 + (1-dx_a1)*dy_a1*v01_a1 + dx_a1*dy_a1*v11_a1
print("bilinear sample:", round(interp_a1, 3))
assert round(interp_a1, 3) == 7.8

▶ What you'll see: fractional sampling blends four neighbors instead of snapping to one.

In [ ]:
print("absolute rounding error:", round(abs(rounded_a1 - interp_a1), 3))
assert round(abs(rounded_a1 - interp_a1), 3) == 1.2

▶ What you'll see: a small coordinate shift changes the sampled feature by 1.2 units.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(F_a1, cmap="viridis")
plt.scatter([point_a1[0]], [point_a1[1]], c="red", label="fractional RoI point")
plt.colorbar(label="feature"); plt.legend(); plt.title("Advanced 1: alignment matters"); plt.show()

▶ What you'll see: the red point lies between feature cells, which is why RoIAlign avoids early rounding.

👀 Takeaway: RoIAlign was introduced because rounding errors can be large relative to small objects.

### Advanced 2 — Encode and decode box regression targets

**Goal.** Use center/size regression targets, because real detectors predict normalized shifts and log-scale changes rather than raw corner offsets. We build it in 4 steps.

In [ ]:
anchor_a2 = np.array([0., 0., 4., 2.])
target_a2 = np.array([1., 0.5, 5., 3.5])
def center_size_a2(b_a2):
    w_a2 = b_a2[2] - b_a2[0]; h_a2 = b_a2[3] - b_a2[1]
    return np.array([(b_a2[0]+b_a2[2])/2, (b_a2[1]+b_a2[3])/2, w_a2, h_a2])
a_a2 = center_size_a2(anchor_a2); t_a2 = center_size_a2(target_a2)
print("anchor center/size:", a_a2)
print("target center/size:", t_a2)

▶ What you'll see: the target center moves right/down and the target height grows.

In [ ]:
tx_a2 = (t_a2[0] - a_a2[0]) / a_a2[2]
ty_a2 = (t_a2[1] - a_a2[1]) / a_a2[3]
tw_a2 = np.log(t_a2[2] / a_a2[2])
th_a2 = np.log(t_a2[3] / a_a2[3])
delta_a2 = np.array([tx_a2, ty_a2, tw_a2, th_a2])
print("encoded delta:", np.round(delta_a2, 3))
assert np.allclose(np.round(delta_a2, 3), [0.25, 0.5, 0.0, 0.405])

▶ What you'll see: translation is measured relative to anchor size and height growth is log(1.5).

In [ ]:
pred_cx_a2 = delta_a2[0] * a_a2[2] + a_a2[0]
pred_cy_a2 = delta_a2[1] * a_a2[3] + a_a2[1]
pred_w_a2 = np.exp(delta_a2[2]) * a_a2[2]
pred_h_a2 = np.exp(delta_a2[3]) * a_a2[3]
decoded_a2 = np.array([pred_cx_a2 - pred_w_a2/2, pred_cy_a2 - pred_h_a2/2, pred_cx_a2 + pred_w_a2/2, pred_cy_a2 + pred_h_a2/2])
print("decoded box:", np.round(decoded_a2, 3))
assert np.allclose(decoded_a2, target_a2)

▶ What you'll see: decoding the normalized targets reconstructs the target box exactly.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["tx", "ty", "tw", "th"], delta_a2, color="purple")
plt.title("Advanced 2: normalized box regression target"); plt.show()

▶ What you'll see: translation and scale corrections live on comparable numeric ranges.

👀 Takeaway: normalized regression targets make learning less sensitive to absolute box size.

### Advanced 3 — Build a tiny RPN objectness map

**Goal.** Score anchors at feature-map locations, because Faster R-CNN proposes boxes with a convolutional objectness head. We build it in 4 steps.

In [ ]:
feature_a3 = np.array([[0.1, 0.4, 0.2], [0.3, 1.2, 0.5], [0.2, 0.6, 0.1]])
anchor_sizes_a3 = np.array([1.0, 2.0])
print("feature map:\n", feature_a3)
print("anchor sizes:", anchor_sizes_a3)

▶ What you'll see: the center location has the strongest visual activation.

In [ ]:
logits_a3 = feature_a3[:, :, None] * np.array([1.5, 1.0]) - np.array([0.2, 0.1])
objectness_a3 = 1 / (1 + np.exp(-logits_a3))
print("objectness shape:", objectness_a3.shape)
print("center scores:", np.round(objectness_a3[1, 1], 3))

▶ What you'll see: each location has two anchor objectness probabilities.

In [ ]:
flat_idx_a3 = int(np.argmax(objectness_a3))
y_a3, x_a3, k_a3 = np.unravel_index(flat_idx_a3, objectness_a3.shape)
print("best location/anchor:", (y_a3, x_a3, k_a3), "score:", round(float(objectness_a3[y_a3, x_a3, k_a3]), 3))
assert (y_a3, x_a3) == (1, 1)

▶ What you'll see: the highest objectness proposal comes from the center feature location.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(objectness_a3[:, :, 0], cmap="magma", vmin=0, vmax=1)
plt.colorbar(label="objectness for small anchor")
plt.title("Advanced 3: RPN objectness heatmap"); plt.show()

▶ What you'll see: objectness peaks where the feature activation is strongest.

👀 Takeaway: an RPN is a learned proposal scorer sliding over shared CNN features.

### Advanced 4 — Evaluate proposal recall versus threshold

**Goal.** Measure whether a proposal set covers ground truth, because proposal stages optimize high recall before final classification. We build it in 4 steps.

In [ ]:
def area_a4(b_a4):
    return max(0.0, b_a4[2]-b_a4[0]) * max(0.0, b_a4[3]-b_a4[1])
def iou_a4(a_a4, b_a4):
    inter_a4 = [max(a_a4[0], b_a4[0]), max(a_a4[1], b_a4[1]), min(a_a4[2], b_a4[2]), min(a_a4[3], b_a4[3])]
    ia_a4 = area_a4(inter_a4); denom_a4 = area_a4(a_a4) + area_a4(b_a4) - ia_a4
    return ia_a4 / denom_a4 if denom_a4 > 0 else 0.0
proposals_a4 = np.array([[0., 0., 2., 2.], [1., 1., 4., 4.], [5., 5., 7., 7.], [0., 0., 4., 4.]])
gts_a4 = np.array([[1., 1., 3., 3.], [5., 5., 7., 7.]])
print("proposals:", len(proposals_a4), "ground truths:", len(gts_a4))

▶ What you'll see: four candidate proposals must cover two real objects.

In [ ]:
iou_matrix_a4 = np.array([[iou_a4(p_a4, g_a4) for g_a4 in gts_a4] for p_a4 in proposals_a4])
best_per_gt_a4 = iou_matrix_a4.max(axis=0)
print("IoU matrix:\n", np.round(iou_matrix_a4, 3))
print("best IoU per GT:", np.round(best_per_gt_a4, 3))

▶ What you'll see: each ground truth looks for its best matching proposal.

In [ ]:
thresholds_a4 = np.array([0.3, 0.5, 0.7, 0.9])
recall_a4 = np.array([np.mean(best_per_gt_a4 >= t_a4) for t_a4 in thresholds_a4])
print("recall by threshold:", np.round(recall_a4, 3))
assert recall_a4[0] == 1.0 and recall_a4[-1] == 0.5

▶ What you'll see: stricter IoU thresholds reduce proposal recall.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(thresholds_a4, recall_a4, marker="o", color="seagreen")
plt.ylim(0, 1.05); plt.xlabel("IoU threshold"); plt.ylabel("proposal recall")
plt.title("Advanced 4: proposal recall curve"); plt.show()

▶ What you'll see: recall stays high at loose thresholds and drops when tight localization is required.

👀 Takeaway: proposal quality is about covering objects well enough for the second stage to refine them.

### Advanced 5 — Run a miniature two-stage detector

**Goal.** Combine proposals, classification, regression, and NMS, because the full R-CNN family is the pipeline working end to end. We build it in 5 steps.

In [ ]:
proposals_a5 = np.array([[0., 0., 3., 3.], [0.5, 0.5, 3.5, 3.5], [5., 5., 7., 7.]])
features_a5 = np.array([[0.2, 1.0], [0.1, 0.9], [1.2, 0.1]])
print("proposal count:", len(proposals_a5))
print("region features:\n", features_a5)

▶ What you'll see: each proposal has a tiny two-number feature vector.

In [ ]:
W_cls_a5 = np.array([[0.0, -1.0], [0.8, 1.0]])
logits_a5 = features_a5 @ W_cls_a5.T
probs_a5 = np.exp(logits_a5 - logits_a5.max(axis=1, keepdims=True)); probs_a5 = probs_a5 / probs_a5.sum(axis=1, keepdims=True)
object_scores_a5 = probs_a5[:, 1]
print("object scores:", np.round(object_scores_a5, 3))

▶ What you'll see: proposals 0 and 1 look object-like; proposal 2 is weaker for this class head.

In [ ]:
deltas_a5 = np.array([[0.2, 0.2, 0.2, 0.2], [0.0, 0.0, 0.0, 0.0], [0.1, 0.1, 0.1, 0.1]])
refined_a5 = proposals_a5 + deltas_a5
print("refined boxes:\n", np.round(refined_a5, 2))

▶ What you'll see: each proposal receives a small coordinate correction.

In [ ]:
def area_a5(b_a5):
    return max(0.0, b_a5[2]-b_a5[0]) * max(0.0, b_a5[3]-b_a5[1])
def iou_a5(a_a5, b_a5):
    inter_a5 = [max(a_a5[0], b_a5[0]), max(a_a5[1], b_a5[1]), min(a_a5[2], b_a5[2]), min(a_a5[3], b_a5[3])]
    ia_a5 = area_a5(inter_a5); denom_a5 = area_a5(a_a5) + area_a5(b_a5) - ia_a5
    return ia_a5 / denom_a5 if denom_a5 > 0 else 0.0
order_a5 = list(np.argsort(object_scores_a5)[::-1])
keep_a5 = []
while order_a5:
    cur_a5 = order_a5.pop(0)
    if object_scores_a5[cur_a5] >= 0.5:
        keep_a5.append(cur_a5)
    order_a5 = [j_a5 for j_a5 in order_a5 if iou_a5(refined_a5[cur_a5], refined_a5[j_a5]) <= 0.3]
print("final kept detections:", keep_a5)
assert keep_a5 == [0, 2]

▶ What you'll see: the best overlapping detection survives, and the distant detection also remains.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
for i_a5, box_a5 in enumerate(refined_a5):
    color_a5 = "seagreen" if i_a5 in keep_a5 else "crimson"
    ax.add_patch(plt.Rectangle((box_a5[0], box_a5[1]), box_a5[2]-box_a5[0], box_a5[3]-box_a5[1], fill=False, lw=2, ec=color_a5))
    ax.text(box_a5[0] + 0.05, box_a5[1] + 0.25, f"{i_a5}: {object_scores_a5[i_a5]:.2f}", color=color_a5)
ax.set_xlim(-0.5, 7.5); ax.set_ylim(7.5, -0.5); ax.set_aspect("equal")
ax.set_title("Advanced 5: tiny two-stage detector output"); plt.show()

▶ What you'll see: green boxes are the final detections after classification, refinement, thresholding, and NMS.

👀 Takeaway: the R-CNN family is best understood as proposal generation plus per-region classification, box refinement, and duplicate removal.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

The R-CNN family made detection a two-stage conversation: first ask where objects might be, then classify and refine each region carefully.

R-CNN bridges classification and detection by proposing candidate regions, pooling features for each region, and refining boxes with a second-stage head.

Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
rng = np.random.default_rng(7)

## Build the two-stage step

A two-stage detector computes $h_r=\operatorname{RoIPool}(F,r)$ and then predicts $\hat b_r=r+\Delta b_r$. We verify the lesson's proposal IoU, pooled feature, and perfect refinement.

In [ ]:
def box_area(box):
    w = max(0.0, float(box[2] - box[0]))
    h = max(0.0, float(box[3] - box[1]))
    return w * h


def iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = box_area([x1, y1, x2, y2])
    union = box_area(a) + box_area(b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def nms(boxes, scores, threshold=0.3, sort=True):
    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float)
    if sort:
        order = list(np.argsort(-scores))
    else:
        order = list(range(len(scores)))
    keep = []
    while order:
        current = order.pop(0)
        keep.append(int(current))
        rest = []
        for idx in order:
            overlap = iou(boxes[current], boxes[idx])
            if overlap <= threshold:
                rest.append(idx)
        order = rest
    return keep


def ap_from_pr(precision, recall):
    precision = np.array(precision, dtype=float)
    recall = np.array(recall, dtype=float)
    prev = np.r_[0.0, recall[:-1]]
    return float(np.sum(precision * (recall - prev)))


def match_mean_iou(pred_boxes, true_boxes):
    pred_boxes = [np.array(b, dtype=float) for b in pred_boxes]
    true_boxes = [np.array(b, dtype=float) for b in true_boxes]
    if not pred_boxes or not true_boxes:
        return 0.0
    scores = []
    used = set()
    for true_box in true_boxes:
        best = 0.0
        best_idx = -1
        for idx, pred_box in enumerate(pred_boxes):
            if idx in used:
                continue
            score = iou(pred_box, true_box)
            if score > best:
                best = score
                best_idx = idx
        if best_idx >= 0:
            used.add(best_idx)
        scores.append(best)
    return float(np.mean(scores))


def make_scene(size, true_boxes, pred_boxes, scores, seed):
    image = np.zeros((size, size), dtype=float)
    yy, xx = np.mgrid[0:size, 0:size]
    for idx, box in enumerate(true_boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        image[y1:y2, x1:x2] = 0.45 + 0.1 * idx
    noise = np.random.default_rng(seed).normal(0.0, 0.035, size=(size, size))
    image = np.clip(image + noise, 0.0, 1.0)
    return {
        "image": image,
        "true_boxes": np.array(true_boxes, dtype=float),
        "pred_boxes": np.array(pred_boxes, dtype=float),
        "scores": np.array(scores, dtype=float),
    }


def load_detection_ladder():
    rungs = []
    rungs.append(("D1 tiny hand scene", make_scene(8, [[0, 0, 3, 3]], [[1, 1, 4, 4]], [0.9], 1)))
    rungs.append(("D2 two clean boxes", make_scene(16, [[1, 2, 6, 8], [10, 9, 14, 14]], [[1, 2, 6, 8], [9, 8, 14, 14], [0, 1, 6, 7]], [0.95, 0.72, 0.45], 2)))
    rungs.append(("D3 crowded small boxes", make_scene(24, [[2, 2, 7, 8], [9, 3, 15, 9], [15, 14, 21, 21]], [[2, 2, 7, 8], [8, 3, 15, 10], [14, 13, 22, 21], [3, 3, 8, 9]], [0.93, 0.82, 0.74, 0.50], 3)))
    rungs.append(("D4 occlusion and scale", make_scene(32, [[2, 3, 8, 10], [9, 6, 18, 18], [19, 4, 29, 12], [21, 20, 30, 29]], [[1, 3, 9, 11], [8, 6, 18, 17], [18, 3, 30, 13], [20, 18, 31, 30], [10, 7, 18, 18]], [0.88, 0.84, 0.70, 0.66, 0.42], 4)))
    rungs.append(("D5 many noisy overlaps", make_scene(40, [[2, 2, 8, 9], [7, 6, 15, 17], [15, 4, 24, 13], [26, 5, 35, 16], [5, 24, 15, 35], [22, 23, 36, 36]], [[1, 1, 9, 10], [6, 5, 16, 18], [14, 5, 25, 14], [24, 4, 36, 17], [6, 23, 16, 35], [20, 21, 37, 37], [7, 7, 15, 17], [25, 5, 34, 16]], [0.86, 0.81, 0.75, 0.69, 0.58, 0.54, 0.62, 0.51], 5)))
    return rungs


def draw_boxes(ax, scene, pred_boxes, title):
    ax.imshow(scene["image"], cmap="gray", vmin=0.0, vmax=1.0)
    for box in scene["true_boxes"]:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
    for box in pred_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linestyle="--", linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def roi_pool(feature, box, mode="max"):
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = feature[y1:y2, x1:x2]
    if mode == "mean":
        return float(np.mean(crop))
    return float(np.max(crop))


def two_stage_detect():
    proposal = np.array([0, 0, 3, 3], dtype=float)
    target = np.array([1, 1, 4, 4], dtype=float)
    proposal_iou = iou(proposal, target)
    feature = np.array([[1, 3], [2, 4]], dtype=float)
    pooled_max = roi_pool(feature, [0, 0, 2, 2], mode="max")
    pooled_mean = roi_pool(feature, [0, 0, 2, 2], mode="mean")
    refined = proposal + np.array([1, 1, 1, 1], dtype=float)
    refined_iou = iou(refined, target)
    return proposal_iou, pooled_max, pooled_mean, refined, refined_iou

proposal_iou, pooled_max, pooled_mean, refined, refined_iou = two_stage_detect()
print(round(proposal_iou, 3), pooled_max, pooled_mean, refined.tolist(), refined_iou)
assert round(proposal_iou, 3) == 0.286
assert pooled_max == 4.0
assert pooled_mean == 2.5
assert refined.tolist() == [1.0, 1.0, 4.0, 4.0]
assert refined_iou == 1.0

The scene method treats synthetic predictions as proposals, refines each proposal toward its closest object, then applies second-stage NMS.

In [ ]:
def refine_toward_truth(box, true_boxes, strength=0.55):
    overlaps = np.array([iou(box, true_box) for true_box in true_boxes])
    target = true_boxes[int(np.argmax(overlaps))]
    return box + strength * (target - box)


def run_scene(scene):
    refined = np.array([refine_toward_truth(box, scene["true_boxes"]) for box in scene["pred_boxes"]])
    keep = nms(refined, scene["scores"], threshold=0.3, sort=True)
    pred_boxes = refined[keep]
    metric = match_mean_iou(pred_boxes, scene["true_boxes"])
    return pred_boxes, metric

## Synthetic geometry ladder

These detection scenes replace the image-classification ladder because the topic is about boxes, overlap, and ranking. D1 is tiny and hand-checkable; D5 has many boxes, occlusion, and noisy duplicates.

In [ ]:
def box_area(box):
    w = max(0.0, float(box[2] - box[0]))
    h = max(0.0, float(box[3] - box[1]))
    return w * h


def iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = box_area([x1, y1, x2, y2])
    union = box_area(a) + box_area(b) - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def nms(boxes, scores, threshold=0.3, sort=True):
    boxes = np.array(boxes, dtype=float)
    scores = np.array(scores, dtype=float)
    if sort:
        order = list(np.argsort(-scores))
    else:
        order = list(range(len(scores)))
    keep = []
    while order:
        current = order.pop(0)
        keep.append(int(current))
        rest = []
        for idx in order:
            overlap = iou(boxes[current], boxes[idx])
            if overlap <= threshold:
                rest.append(idx)
        order = rest
    return keep


def ap_from_pr(precision, recall):
    precision = np.array(precision, dtype=float)
    recall = np.array(recall, dtype=float)
    prev = np.r_[0.0, recall[:-1]]
    return float(np.sum(precision * (recall - prev)))


def match_mean_iou(pred_boxes, true_boxes):
    pred_boxes = [np.array(b, dtype=float) for b in pred_boxes]
    true_boxes = [np.array(b, dtype=float) for b in true_boxes]
    if not pred_boxes or not true_boxes:
        return 0.0
    scores = []
    used = set()
    for true_box in true_boxes:
        best = 0.0
        best_idx = -1
        for idx, pred_box in enumerate(pred_boxes):
            if idx in used:
                continue
            score = iou(pred_box, true_box)
            if score > best:
                best = score
                best_idx = idx
        if best_idx >= 0:
            used.add(best_idx)
        scores.append(best)
    return float(np.mean(scores))


def make_scene(size, true_boxes, pred_boxes, scores, seed):
    image = np.zeros((size, size), dtype=float)
    yy, xx = np.mgrid[0:size, 0:size]
    for idx, box in enumerate(true_boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        image[y1:y2, x1:x2] = 0.45 + 0.1 * idx
    noise = np.random.default_rng(seed).normal(0.0, 0.035, size=(size, size))
    image = np.clip(image + noise, 0.0, 1.0)
    return {
        "image": image,
        "true_boxes": np.array(true_boxes, dtype=float),
        "pred_boxes": np.array(pred_boxes, dtype=float),
        "scores": np.array(scores, dtype=float),
    }


def load_detection_ladder():
    rungs = []
    rungs.append(("D1 tiny hand scene", make_scene(8, [[0, 0, 3, 3]], [[1, 1, 4, 4]], [0.9], 1)))
    rungs.append(("D2 two clean boxes", make_scene(16, [[1, 2, 6, 8], [10, 9, 14, 14]], [[1, 2, 6, 8], [9, 8, 14, 14], [0, 1, 6, 7]], [0.95, 0.72, 0.45], 2)))
    rungs.append(("D3 crowded small boxes", make_scene(24, [[2, 2, 7, 8], [9, 3, 15, 9], [15, 14, 21, 21]], [[2, 2, 7, 8], [8, 3, 15, 10], [14, 13, 22, 21], [3, 3, 8, 9]], [0.93, 0.82, 0.74, 0.50], 3)))
    rungs.append(("D4 occlusion and scale", make_scene(32, [[2, 3, 8, 10], [9, 6, 18, 18], [19, 4, 29, 12], [21, 20, 30, 29]], [[1, 3, 9, 11], [8, 6, 18, 17], [18, 3, 30, 13], [20, 18, 31, 30], [10, 7, 18, 18]], [0.88, 0.84, 0.70, 0.66, 0.42], 4)))
    rungs.append(("D5 many noisy overlaps", make_scene(40, [[2, 2, 8, 9], [7, 6, 15, 17], [15, 4, 24, 13], [26, 5, 35, 16], [5, 24, 15, 35], [22, 23, 36, 36]], [[1, 1, 9, 10], [6, 5, 16, 18], [14, 5, 25, 14], [24, 4, 36, 17], [6, 23, 16, 35], [20, 21, 37, 37], [7, 7, 15, 17], [25, 5, 34, 16]], [0.86, 0.81, 0.75, 0.69, 0.58, 0.54, 0.62, 0.51], 5)))
    return rungs


def draw_boxes(ax, scene, pred_boxes, title):
    ax.imshow(scene["image"], cmap="gray", vmin=0.0, vmax=1.0)
    for box in scene["true_boxes"]:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
        ax.add_patch(rect)
    for box in pred_boxes:
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linestyle="--", linewidth=1.5)
        ax.add_patch(rect)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def refine_toward_truth(box, true_boxes, strength=0.55):
    overlaps = np.array([iou(box, true_box) for true_box in true_boxes])
    target = true_boxes[int(np.argmax(overlaps))]
    return box + strength * (target - box)


def run_scene(scene):
    refined = np.array([refine_toward_truth(box, scene["true_boxes"]) for box in scene["pred_boxes"]])
    keep = nms(refined, scene["scores"], threshold=0.3, sort=True)
    pred_boxes = refined[keep]
    metric = match_mean_iou(pred_boxes, scene["true_boxes"])
    return pred_boxes, metric

rungs = load_detection_ladder()
for name, scene in rungs:
    print(name, "image", scene["image"].shape, "truth", len(scene["true_boxes"]), "pred", len(scene["pred_boxes"]))

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (name, scene) in zip(axes, rungs):
    draw_boxes(ax, scene, scene["pred_boxes"], name.split()[0])
plt.tight_layout()
plt.show()

## Run the same method across D1–D5

Each rung reports mean best-match IoU.

In [ ]:
results = []
outputs = []
for name, scene in rungs:
    pred_boxes, metric = run_scene(scene)
    results.append((name, metric))
    outputs.append(pred_boxes)

print("rung                         metric")
for name, metric in results:
    print(f"{name:28s} {metric:.3f}")

## Results visualization

Green boxes are truth, dashed red boxes are the method output. The curve shows localization quality as complexity rises.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (name, scene), pred_boxes in zip(axes[0], rungs, outputs):
    draw_boxes(ax, scene, pred_boxes, name.split()[0])

xs = np.arange(1, 6)
ys = [metric for name, metric in results]
axes[1, 0].plot(xs, ys, marker="o")
axes[1, 0].set_xticks(xs)
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].set_xlabel("complexity rung")
axes[1, 0].set_ylabel("IoU/AP metric")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5

Treating proposals as final detections and rounding RoIs too early hurts small objects. Refinement and RoIAlign-style fractional coordinates improve the hardest scene.

In [ ]:
name, scene = rungs[-1]
proposal_metric = match_mean_iou(scene["pred_boxes"], scene["true_boxes"])
rounded = np.round(scene["pred_boxes"])
rounded_metric = match_mean_iou(rounded, scene["true_boxes"])
refined_boxes, refined_metric = run_scene(scene)
print("proposal-only metric", round(proposal_metric, 3))
print("rounded proposal metric", round(rounded_metric, 3))
print("refined fractional metric", round(refined_metric, 3))

## Evaluate it + Practice

- Compare the reported IoU with a no-skill baseline that predicts one large center box or all background.
- Overfit D1: the hand scene should reproduce the exact lesson arithmetic before scaling up.
- Ablate the key idea, such as sorting before NMS, refinement, objectness, focal weighting, matching, or nearest-neighbor masks.
- Watch for failure signals: duplicate boxes, high pixel accuracy with poor IoU, and metrics that improve only because the scene got easier.

Practice:
1. Change the D3 object positions and predict how IoU changes.
2. Tighten the matching threshold and rerun the table.
3. Add one noisy false positive to D5 and explain the curve.